Goal: Understand time unrolling, hidden-state flow, and gradient issues in Vanilla RNN


In [32]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [33]:
# =====================
# Hyperparameters
# =====================
seq_len = 25
batch_size = 32
hidden_size = 128
lr = 1e-3
epochs = 20


In [34]:
with open("data/input.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Optional: limit size for fast experiments
text = text[:200_000]

chars = sorted(list(set(text)))
vocab_size = len(chars)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

encoded = torch.tensor([char_to_idx[ch] for ch in text], dtype=torch.long)

print("Dataset size:", len(encoded))
print("Vocab size:", vocab_size)


Dataset size: 200000
Vocab size: 83


In [35]:
def get_batch(encoded, batch_size, seq_len):
    N = encoded.size(0)
    start_idx = torch.randint(0, N - seq_len - 1, (batch_size,))

    X = torch.stack([encoded[i:i+seq_len] for i in start_idx])
    Y = torch.stack([encoded[i+1:i+seq_len+1] for i in start_idx])

    return X, Y

In [36]:
def one_hot(x, vocab_size):
    return torch.zeros(x.size(0), vocab_size).scatter_(1, x.unsqueeze(1), 1)


In [37]:
class VanillaRNNCell(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size

        self.W_xh = nn.Parameter(torch.randn(vocab_size, hidden_size) * 0.01)
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
        self.b_h  = nn.Parameter(torch.zeros(hidden_size))

        self.W_hy = nn.Parameter(torch.randn(hidden_size, vocab_size) * 0.01)
        self.b_y  = nn.Parameter(torch.zeros(vocab_size))

    def forward(self, x_t, h_prev):
        x_onehot = one_hot(x_t, self.W_xh.size(0))
        h_t = torch.tanh(x_onehot @ self.W_xh + h_prev @ self.W_hh + self.b_h)
        y_t = h_t @ self.W_hy + self.b_y
        return h_t, y_t


In [38]:
def forward_rnn(cell, X):
    B, T = X.shape
    h = torch.zeros(B, cell.hidden_size)

    logits = []
    for t in range(T):
        h, y_t = cell(X[:, t], h)
        logits.append(y_t)

    return torch.stack(logits, dim=1)  # (B, T, V)


In [39]:
cell = VanillaRNNCell(vocab_size, hidden_size=hidden_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cell.parameters(), lr=lr)

In [40]:
for epoch in range(epochs):
    epoch_loss = 0

    for _ in range(100):
        X, Y = get_batch(encoded, batch_size , seq_len)

        logits = forward_rnn(cell, X)

        loss = criterion(
            logits.reshape(-1, vocab_size),
            Y.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(cell.parameters(), 5)
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {epoch_loss/100:.4f}")


Epoch 1, Loss: 3.4603
Epoch 2, Loss: 3.2660
Epoch 3, Loss: 3.2527
Epoch 4, Loss: 3.2368
Epoch 5, Loss: 3.1908
Epoch 6, Loss: 3.1166
Epoch 7, Loss: 3.0255
Epoch 8, Loss: 2.9423
Epoch 9, Loss: 2.8436
Epoch 10, Loss: 2.7526
Epoch 11, Loss: 2.6618
Epoch 12, Loss: 2.6000
Epoch 13, Loss: 2.5361
Epoch 14, Loss: 2.4680
Epoch 15, Loss: 2.4063
Epoch 16, Loss: 2.3410
Epoch 17, Loss: 2.2926
Epoch 18, Loss: 2.2399
Epoch 19, Loss: 2.2138
Epoch 20, Loss: 2.1696


In [41]:
def generate_text(cell, start_char, length=300, temperature=1.0):
    cell.eval()

    idx = torch.tensor([char_to_idx[start_char]])
    h = torch.zeros(1, cell.hidden_size)

    generated = [start_char]

    for _ in range(length):
        h, logits = cell(idx, h)
        logits = logits / temperature

        probs = F.softmax(logits.squeeze(0), dim=-1)
        idx = torch.multinomial(probs, 1)

        generated.append(idx_to_char[idx.item()])

    return "".join(generated)


In [42]:
print(generate_text(cell, start_char="T", length=500))


THELOUNI.
Thim motetald fe; ind I shivitht te baigulesid serw

Woray!
L] LOU4-Anz seofeyrs  lyous nive.


ARFIRN.
Th unuth the men, hite come yly men, meve,
Thow dines’tt oilhe,
Shis wer wond I all noflon shouts lores  all ow ell.

  Momunciste,
mdy’tht thee halles e stabe’t  Rill igugr hy surrvuir._
UEWOSSES.
Whes cyout yelnen houswen moth of az’st you por anist mlen hors.

PAnMpy inghey cotlorinof shas, a wave,
Thin metoll and mue’n  null remeires cat,
Sold hartt,
 hath aks dast fanes: wiimt th


In [43]:
vocab_size = 83 ; hidden_size = 128 
import torch 
torch.randn(vocab_size, hidden_size).shape 

torch.Size([83, 128])